# 09 — Build a bulk cross-run analysis

This notebook runs the independent post-processing pipeline over a directory tree of completed SpectralBridge outputs. It creates a provenance-preserving super Parquet, a DuckDB catalog, and pooled synthetic MicaSense-to-Landsat regression coefficients without changing individual NEON or drone runs.

## 1. Configure the source tree and output directory

The default `input_kind="full"` excludes polygon subsets so they are not counted twice when a flightline has both full and polygon merged products. Use an absolute path under your home directory or another mounted data location when appropriate.

In [ ]:
from pathlib import Path
from pprint import pprint

import duckdb

from spectralbridge import run_bulk_pipeline

RUN = False
input_root = Path.home() / "processed_spectralbridge"
output_dir = Path.home() / "spectralbridge_bulk_results"
input_kind = "full"
memory_limit = "8GB"
threads = 4

## 2. Build or reuse the collection

The pipeline recursively inventories canonical merged Parquets. It rebuilds when the source inventory or analysis settings change and otherwise reports `status="reused"`.

In [ ]:
result = None
if RUN:
    result = run_bulk_pipeline(
        input_root,
        output_dir,
        input_kind=input_kind,
        memory_limit=memory_limit,
        threads=threads,
    )
    pprint(result)
else:
    print("Edit the paths and set RUN = True to build the bulk collection.")

## 3. Inspect sources and coefficients

The database stores the source catalog and pooled coefficient table. Its `bulk_observations` view reads the portable super Parquet without storing a second full copy.

In [ ]:
if RUN and result is not None:
    with duckdb.connect(result["database"], read_only=True) as con:
        source_summary = con.execute(
            "SELECT status, COUNT(*) AS files, SUM(row_count) AS rows "
            "FROM bulk_sources GROUP BY status ORDER BY status"
        ).df()
        coefficients = con.execute(
            "SELECT landsat_sensor, band_index, slope, intercept, r2, "
            "sample_count, source_count "
            "FROM synthetic_translation_coefficients "
            "ORDER BY landsat_sensor, band_index"
        ).df()
    display(source_summary)
    display(coefficients)

## 4. Interpret the result

Each equation is `Landsat = slope × MicaSense + intercept` and is fitted from all valid persisted rows. These slopes and intercepts are separate from the fixed percentage brightness adjustment. Larger source tables contribute more observations, and the same-source synthetic regressions are descriptive until reviewed for their intended downstream use.

In [ ]:
for artifact in sorted(output_dir.glob("*")):
    print(artifact.name, artifact.stat().st_size)